In [1]:
import torch
import torch.nn as nn
import yaml
import h5py

import numpy as np

from models.fno import FNO1d
from models.pitt import PhysicsInformedTokenTransformer
from utils import TransformerOperatorDataset

device = 'cuda' if(torch.cuda.is_available()) else 'cpu'

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [2]:
with open("./configs/pitt_config.yaml", 'r') as stream:
        config = yaml.safe_load(stream)

train_args = config['args']
prefix = train_args['flnm'] + "_" + train_args['data_name'].split("_")[0] + "_" + train_args['train_style'] + "_" + \
             train_args['embedding']
train_args['prefix'] = prefix

In [3]:
neural_operator = FNO1d(train_args['num_channels'], train_args['modes'], train_args['width'], train_args['initial_step'], train_args['dropout'])

In [4]:
transformer = PhysicsInformedTokenTransformer(500, train_args['hidden'], train_args['layers'], train_args['heads'],
                                    train_args['num_x'], dropout=train_args['dropout'], neural_operator=neural_operator).to(device=device)

In [5]:
def get_data(f, config):
    test_data = TransformerOperatorDataset(f, config['flnm'],
                            split="test",
                            initial_step=config['initial_step'],
                            reduced_resolution=config['reduced_resolution'],
                            reduced_resolution_t=config['reduced_resolution_t'],
                            reduced_batch=config['reduced_batch'],
                            saved_folder=config['base_path'],
                            return_text=config['return_text'],
                            num_t=config['num_t'],
                            num_x=config['num_x'],
                            sim_time=config['sim_time'],
                            num_samples=config['num_samples'],
                            train_style=config['train_style'],
                            rollout_length=config['rollout_length'],
                            interval=config['interval'],
    )
    test_data.data = test_data.data.to(device)
    test_data.grid = test_data.grid.to(device)

    test_loader = torch.utils.data.DataLoader(test_data, batch_size=config['batch_size'],
                                             num_workers=config['num_workers'], shuffle=False,
                                             generator=torch.Generator(device=device))
    
    return test_loader


In [6]:
def evaluate(test_loader, transformer, loss_fn):
    #src_mask = generate_square_subsequent_mask(640).cuda()
    with torch.no_grad():
        transformer.eval()
        test_loss = 0
        for bn, (x0, y, grid, tokens, t) in enumerate(test_loader):

            y_pred = transformer(grid.to(device=device), tokens.to(device=device), x0.to(device=device), t.to(device=device))

            y = y[...,0].to(device=device)

            # Compute the loss.
            test_loss += loss_fn(y_pred, y).item()
    return test_loss/(bn+1)

In [7]:
loss_list = []
for seed in range(5):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_path = f"1D_results_time_cont/pitt_fno_Heat_varied_interpolate_novel/model_param_{seed}.pt"
    
    f = h5py.File("{}{}".format(train_args['base_path'], train_args['data_name']), 'r')
    test_loader = get_data(f, train_args)

    loss_fn = nn.MSELoss(reduction='mean')

    transformer.load_state_dict(torch.load(model_path)['model_param'])
    test_value = evaluate(test_loader, transformer, loss_fn)
    print(f'Loss test set seed {seed}:', test_value)
    loss_list.append(test_value)


SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:15<00:00, 786.52it/s]



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1078.55it/s]
/home/13422855/code/PhysicsInformedTokenTransformer/utils.py:489: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/torch/csrc/utils/tensor_new.cpp:278.)
  self.time = torch.Tensor(self.time).to(device=device)
/local_scratch/ipykernel_237859/2834086676.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpicklin

Loss test set seed 0: 0.0011503735013514203

SEED: 0
Gathering data...


...

Loss test set seed 3: 0.0011504067955142323

SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:15<00:00, 756.57it/s]



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1071.84it/s]


Loss test set seed 4: 0.02450248805132318


In [8]:
print(loss_list)

[0.0011503735013514203, 0.0011953941213234546, 0.0011712528915055335, 0.0011504067955142323, 0.02450248805132318]


In [9]:
loss_list_end = []
for seed in range(5):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_path = f"1D_results_time_cont/pitt_fno_Heat_varied_interpolate_novel/model_param_end_{seed}.pt"
    
    f = h5py.File("{}{}".format(train_args['base_path'], train_args['data_name']), 'r')
    test_loader = get_data(f, train_args)

    loss_fn = nn.MSELoss(reduction='mean')

    transformer.load_state_dict(torch.load(model_path)['model_param'])
    test_value = evaluate(test_loader, transformer, loss_fn)
    print(f'Loss test set seed {seed}:', test_value)
    loss_list_end.append(test_value)


SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:17<00:00, 698.13it/s]



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1073.60it/s]
/local_scratch/ipykernel_237859/1071072472.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  transformer.load_state_dict(torch.load(model_pat

Loss test set seed 0: 0.0011315293482534509

SEED: 0
Gathering data...


...

Loss test set seed 3: 0.0011508803005934633

SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:19<00:00, 631.38it/s]



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1073.11it/s]


Loss test set seed 4: 0.023992160117214032


In [10]:
print(loss_list_end)

[0.0011315293482534509, 0.0011908049308793976, 0.001171350814129642, 0.0011508803005934633, 0.023992160117214032]


In [12]:
import csv

step = train_args['initial_step']
interval = train_args['interval']

with open(f'1D_results_time_cont/pitt_fno_Heat_varied_interpolate_novel/test_vals_step{step}_int{interval}.csv', mode ='r')as file:
          csvFile = csv.reader(file)
          loss = []
          for line in csvFile:
              line = [float(i) for i in line]
              loss.append(line)

print('test loss', loss[0])
print('best loss', loss[3])

test loss [0.001131184381789508, 0.001216130885989108, 0.0011847604058702734, 0.0011507746629773618, 0.023464294447702295]
best loss [0.0011284942682981095, 0.0012058713281418176, 0.0011720137349874812, 0.001138920531647795, 0.023948072871946273]
